In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import precision_score, accuracy_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from itertools import product

In [8]:
data = pd.read_csv('C:\\Users\\Ryan George\\OneDrive\\Desktop\\study\\Prem_pred\\proj3\\match_df.csv')

In [9]:
#fix opponent name
opponent_name_fix = {
    "Manchester Utd": "Manchester United",
    "Newcastle Utd": "Newcastle United",
    "Brighton": "Brighton and Hove Albion",
    "West Brom": "West Bromwich Albion",
    "West Ham": "West Ham United",
    "Wolves": "Wolverhampton Wanderers",
    "Tottenham": "Tottenham Hotspur",
    "Sheffield Utd": "Sheffield United",
    "Nott'ham Forest": "Nottingham Forest",
    "Huddersfield": "Huddersfield Town",
}
data["opponent"] = data["opponent"].replace(opponent_name_fix)

In [10]:
cols_to_keep = [
    'date', 'time', 'round', 'day', 'venue', 'result',
    'gf', 'ga_x', 'opponent', 'xg_x', 'xga', 'poss_x',
    'captain', 'formation', 'opp formation', 'referee',
    'sh_x', 'sot', 'crdy', 'crdr', '2crdy', 'season', 'team'
]
data = data[cols_to_keep].copy()

rename_map = {"ga_x": "ga", "xg_x": "xg", "poss_x": "poss", "sh_x": "sh"}
data = data.rename(columns=rename_map)

In [11]:
#features
data["date"]       = pd.to_datetime(data["date"], format="%d-%m-%Y")
data["target"]     = (data["result"] == "W").astype(int)
data["venue_code"] = data["venue"].astype("category").cat.codes
data["opp_code"]   = data["opponent"].astype("category").cat.codes
data["hour"]       = data["time"].str.replace(":.+", "", regex=True).astype(int)
data["day_code"]   = data["date"].dt.dayofweek
data["matchweek"]  = data["round"].str.extract(r"(\d+)").astype(int)

In [12]:
#points and gd
data = data.sort_values(["team", "season", "date"])
data["gd"] = data["gf"] - data["ga"]

data["points"] = 0
data.loc[data["result"] == "W", "points"] = 3
data.loc[data["result"] == "D", "points"] = 1

In [13]:
#league position
def add_true_league_position(df):
    df = df.sort_values(["season", "date", "time"]).copy()
    df["league_position"] = np.nan

    for season, season_df in df.groupby("season"):
        teams = season_df["team"].unique()
        table = {team: {"points": 0, "gd": 0, "gf": 0} for team in teams}

        for matchweek, mw_df in season_df.groupby("matchweek"):
            ranked = sorted(
                table.items(),
                key=lambda x: (x[1]["points"], x[1]["gd"], x[1]["gf"]),
                reverse=True
            )
            pos_map = {team: i+1 for i, (team, _) in enumerate(ranked)}
            df.loc[mw_df.index, "league_position"] = mw_df["team"].map(pos_map)

            for _, row in mw_df.iterrows():
                table[row["team"]]["points"] += row["points"]
                table[row["team"]]["gd"]     += row["gd"]
                table[row["team"]]["gf"]     += row["gf"]

    return df

data = add_true_league_position(data)

In [14]:
#stats before
data = data.sort_values(["team", "season", "date"]).reset_index(drop=True)

data["gd_before"] = (
    data.groupby(["team", "season"])["gd"].cumsum().shift(1).fillna(0).astype(int)
)
data["points_before"] = (
    data.groupby(["team", "season"])["points"].cumsum().shift(1).fillna(0).astype(int)
)
data["gf_before"] = (
    data.groupby(["team", "season"])["gf"].cumsum().shift(1).fillna(0).astype(int)
)
data["ga_before"] = (
    data.groupby(["team", "season"])["ga"].cumsum().shift(1).fillna(0).astype(int)
)

In [15]:
#form
def compute_form(group, n=5):
    group = group.sort_values("date")
    group[f"wins_last{n}"]   = group["target"].rolling(n, closed="left").sum().fillna(0)
    group[f"draws_last{n}"]  = (group["result"] == "D").astype(int).rolling(n, closed="left").sum().fillna(0)
    group[f"losses_last{n}"] = (group["result"] == "L").astype(int).rolling(n, closed="left").sum().fillna(0)
    return group

data = (
    data.reset_index(drop=True)
    .set_index(["team", "season"])
    .groupby(["team", "season"], group_keys=False)
    .apply(compute_form)
    .reset_index()
)

In [16]:
#home away form
def home_away_form(group, n=5):
    group = group.sort_values("date")
    for venue_type, prefix in [("Home", "home"), ("Away", "away")]:
        mask = group["venue"] == venue_type
        col  = f"{prefix}_wins_last{n}"
        venue_wins = group["target"].where(mask)
        group[col] = (
            venue_wins
            .rolling(n, min_periods=1, closed="left")
            .sum()
            .where(mask)
            .ffill()
            .fillna(0)
        )
    return group

data = (
    data.reset_index(drop=True)
    .set_index(["team", "season"])
    .groupby(["team", "season"], group_keys=False)
    .apply(home_away_form)
    .reset_index()
)

In [17]:
#days
data = data.sort_values(["team", "date"])
data["days_since_last"] = (
    data.groupby("team")["date"]
    .diff().dt.days.fillna(7).astype(int)
)

In [18]:
#streak
def compute_streak(group):
    group = group.sort_values("date")
    streaks = []
    current_streak = 0
    for result in group["result"].shift(1):
        if pd.isna(result):
            streaks.append(0)
        elif result == "W":
            current_streak = max(1, current_streak + 1) if current_streak >= 0 else 1
            streaks.append(current_streak)
        elif result == "L":
            current_streak = min(-1, current_streak - 1) if current_streak <= 0 else -1
            streaks.append(current_streak)
        else:
            current_streak = 0
            streaks.append(0)
    group["streak"] = streaks
    return group

data = (
    data.reset_index(drop=True)
    .set_index(["team", "season"])
    .groupby(["team", "season"], group_keys=False)
    .apply(compute_streak)
    .reset_index()
)

In [19]:
#clean sheet
def compute_clean_sheets(group, n=5):
    group = group.sort_values("date")
    group[f"clean_sheets_last{n}"] = (
        (group["ga"] == 0).astype(int)
        .rolling(n, closed="left")
        .sum()
        .fillna(0)
    )
    return group

data = (
    data.reset_index(drop=True)
    .set_index(["team", "season"])
    .groupby(["team", "season"], group_keys=False)
    .apply(compute_clean_sheets)
    .reset_index()
)

In [20]:
#h2h
def compute_h2h(data):
    data = data.sort_values(["team", "opponent", "date"])
    data["h2h_wins"] = (
        data.groupby(["team", "opponent"])["target"]
        .apply(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
        .reset_index(level=[0,1], drop=True).fillna(0)
    )
    data["h2h_games"] = (
        data.groupby(["team", "opponent"])["target"]
        .apply(lambda x: x.shift(1).rolling(5, min_periods=1).count())
        .reset_index(level=[0,1], drop=True).fillna(0)
    )
    data["h2h_win_rate"] = (data["h2h_wins"] / data["h2h_games"]).fillna(0)
    return data

data = compute_h2h(data)

In [21]:
#ref
data["referee_code"] = data["referee"].astype("category").cat.codes

def compute_ppg(group, n=5):
    group = group.sort_values("date")
    group[f"ppg_last{n}"] = (
        group["points"].rolling(n, closed="left").mean().fillna(0)
    )
    return group

data = (
    data.reset_index(drop=True)
    .set_index(["team", "season"])
    .groupby(["team", "season"], group_keys=False)
    .apply(compute_ppg)
    .reset_index()
)

In [22]:
data = data.sort_values(["team", "season", "date"]).reset_index(drop=True)

opp_form = data[[
    "date", "team",
    "wins_last5", "draws_last5", "losses_last5",
    "home_wins_last5", "away_wins_last5",
    "days_since_last", "streak", "clean_sheets_last5",
    "h2h_wins", "h2h_win_rate", "ppg_last5",
    "league_position", "points_before", "gd_before",
    "gf_before", "ga_before",
]].copy()

opp_form.columns = [
    "date", "opponent",
    "opp_wins_last5", "opp_draws_last5", "opp_losses_last5",
    "opp_home_wins_last5", "opp_away_wins_last5",
    "opp_days_since_last", "opp_streak", "opp_clean_sheets_last5",
    "opp_h2h_wins", "opp_h2h_win_rate", "opp_ppg_last5",
    "opp_league_position", "opp_points_before", "opp_gd_before",
    "opp_gf_before", "opp_ga_before",
]

data = data.merge(opp_form, on=["date", "opponent"], how="left")

data["points_diff"]   = data["points_before"] - data["opp_points_before"]
data["gd_diff"]       = data["gd_before"] - data["opp_gd_before"]
data["position_diff"] = data["league_position"] - data["opp_league_position"]

In [23]:
predictors = [
    "venue_code", "opp_code", "hour", "day_code", "matchweek",
    "gd_before", "points_before", "league_position",
    "gf_before", "ga_before",
    "home_wins_last5", "days_since_last", "streak",
    "clean_sheets_last5", "h2h_win_rate", "referee_code", "ppg_last5",
    "opp_days_since_last", "opp_streak", "opp_clean_sheets_last5",
    "opp_h2h_win_rate", "opp_ppg_last5",
    "opp_league_position", "opp_points_before", "opp_gd_before",
    "opp_gf_before", "opp_ga_before",
    "points_diff", "gd_diff", "position_diff",
]

In [24]:
def make_predictions(data, predictors, model):
    train = data[data["season"].isin([2020, 2021, 2022, 2023])]
    test  = data[data["date"] >= "2024-08-16"]

    model.fit(train[predictors], train["target"])
    preds = model.predict(test[predictors])

    combined = pd.DataFrame(
        {"actual": test["target"], "predicted": preds},
        index=test.index
    )
    error    = precision_score(test["target"], preds)
    accuracy = accuracy_score(test["target"], preds)
    return combined, error, accuracy

In [25]:
best_ensemble = VotingClassifier(
    estimators=[
        ("rf",   RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)),
        ("xgb",  XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=1, eval_metric="logloss", verbosity=0)),
        ("gb",   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=1)),
        ("lgbm", LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=1, verbose=-1))
    ],
    voting="soft",
    weights=[19, 16, 1, 6]
)

combined, error, accuracy = make_predictions(data, predictors, best_ensemble)
print(f"Precision: {error:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(classification_report(combined["actual"], combined["predicted"], target_names=["Not Win", "Win"]))

Precision: 0.6318
Accuracy:  0.6921
              precision    recall  f1-score   support

     Not Win       0.71      0.84      0.77       473
         Win       0.63      0.44      0.52       287

    accuracy                           0.69       760
   macro avg       0.67      0.64      0.65       760
weighted avg       0.68      0.69      0.68       760

